# Spark SQL with an External Hive Catalog

This notebook uses Spark SQL with the course Hive Metastore and stores managed-table data in the Hive warehouse on HDFS. It deliberately does **not** use Spark's embedded Derby metastore.

## Learning objectives

- Distinguish catalog metadata from table data.
- Connect Spark to a shared Hive Metastore.
- Create, query, inspect, and remove a managed table.
- Observe the managed table's HDFS location.

## 1. Start the required services

This notebook requires HDFS, the Spark standalone cluster, and the external Hive Metastore described in the course setup notes. Start them in a WSL terminal:

```bash
start-dfs.sh
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"

nohup hive --service metastore \
  > "$HOME/hive-logs/metastore.log" 2>&1 &

jps
ss -lnt | grep ':9083'
hdfs dfsadmin -report
```

Port 9083 must be listening before Spark starts. The external metastore lets separate Spark sessions and Hive clients share catalog metadata. HiveServer2 is not required for Spark to contact the metastore directly.

## 2. Connect Spark to the Hive catalog

`enableHiveSupport()` enables Hive catalog integration. The metastore URI identifies the shared metadata service. Warehouse configuration should come from the Hive/Hadoop configuration installed for the lab.

Do not set `spark.local.dir` to HDFS. Spark local scratch space belongs on the worker's local filesystem and is managed by the Spark installation.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D30-Spark-Hive-Catalog")
    .master(master_url)
    .config("hive.metastore.uris", "thrift://localhost:9083")
    .enableHiveSupport()
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Catalog       :", spark.conf.get("spark.sql.catalogImplementation"))
print("Warehouse     :", spark.conf.get("spark.sql.warehouse.dir"))
print("Spark UI      :", sc.uiWebUrl)

## 3. Verify the catalog connection

The catalog implementation must be `hive`. Existing Hive databases should be visible.

In [ ]:
spark.sql("SHOW DATABASES").show(truncate=False)

## 4. Create a training database

The Hive Metastore stores the database definition. For a managed database, its data directory is created under the configured warehouse, normally `/user/hive/warehouse/spark_training.db`.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS spark_training")
spark.sql("DESCRIBE DATABASE EXTENDED spark_training").show(truncate=False)

## 5. Create a managed table

Spark records the table schema and location in Hive and stores the table data in the warehouse.

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS spark_training.stocks (
    symbol STRING,
    industry STRING
)
USING PARQUET
""")

spark.sql("SHOW TABLES IN spark_training").show()

## 6. Insert and query data

Overwrite makes the lesson repeatable without accumulating duplicate rows.

In [ ]:
spark.sql("""
INSERT OVERWRITE spark_training.stocks VALUES
    ('INFY', 'IT'),
    ('HDFCBANK', 'Banking')
""")

spark.sql("SELECT * FROM spark_training.stocks ORDER BY symbol").show()

## 7. Inspect metadata and HDFS storage

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED spark_training.stocks").show(100, truncate=False)

In [ ]:
%%bash
hdfs dfs -ls -R /user/hive/warehouse/spark_training.db

## 8. Managed-table cleanup

Dropping a managed table removes both its catalog entry and its managed data directory. The commands below intentionally remove only the objects created by this lesson.

In [ ]:
spark.sql("DROP TABLE IF EXISTS spark_training.stocks")
spark.sql("DROP DATABASE IF EXISTS spark_training")
spark.sql("SHOW DATABASES").show(truncate=False)

## 9. Stop Spark

Run this cell when the lesson is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")